# Phân loại email spam — Multinomial Naive Bayes

Bài giảng `Machine_Learning_Full_Course.pdf`, trang 58–60.
Theo ví dụ scikit-learn trang 34–38: đọc → chia tập → xử lý → fit → predict → đánh giá.
TF-IDF và làm sạch văn bản lấy từ đề Lab 1, không gán là nội dung của trang Naive Bayes.
Phần Naive Bayes tự viết công thức; NumPy hỗ trợ tính toán mảng.

## 1. Mã giả

Quy ước: RỖNG nghĩa là None/chưa có trạng thái, khác bảng rỗng {}.
Chỉ số bắt đầu từ 0; V là số đặc trưng TF-IDF. FIT không trả nhãn dự đoán.
Mã giả trình bày vòng lặp từng phần tử; code NumPy gộp một số vòng lặp thành phép tính mảng.

```text
ĐẦU VÀO: X_train (N × V), y_train (N nhãn), X_test (M × V), alpha > 0
ĐẦU RA: y_pred (M nhãn)

PHƯƠNG THỨC INIT(alpha):
    lưu alpha
    classes ← RỖNG; prior ← RỖNG; probability ← RỖNG

PHƯƠNG THỨC BUILD_MODEL():
    NẾU alpha ≤ 0: BÁO LỖI, DỪNG
    classes ← RỖNG
    prior ← bảng rỗng
    probability ← bảng rỗng

PHƯƠNG THỨC FIT(X_train, y_train):
    NẾU prior = RỖNG: BÁO LỖI, DỪNG
    N ← số dòng X_train
    V ← số cột X_train
    classes ← các nhãn khác nhau trong y_train, sắp xếp tăng dần
    VỚI MỖI lớp c TRONG classes:
        indices ← các i thỏa y_train[i] = c
        prior[c] ← số phần tử indices / N
        totals ← mảng V phần tử bằng 0
        VỚI MỖI i TRONG indices:
            VỚI j ← 0 ĐẾN V − 1:
                totals[j] ← totals[j] + X_train[i,j]
        denominator ← tổng(totals) + alpha × V
        VỚI j ← 0 ĐẾN V − 1:
            probability[c,j] ← (totals[j] + alpha) / denominator

HÀM PREDICT(X_test):
    NẾU classes = RỖNG: BÁO LỖI, DỪNG
    y_pred ← danh sách rỗng
    VỚI MỖI vector x TRONG X_test:
        best_label ← classes[0]
        best_score ← −∞
        VỚI MỖI lớp c TRONG classes:
            score ← log(prior[c])
            VỚI j ← 0 ĐẾN số đặc trưng − 1:
                score ← score + x[j] × log(probability[c,j])
            NẾU score > best_score:
                best_score ← score
                best_label ← c
        THÊM best_label VÀO y_pred
    TRẢ VỀ y_pred
```


In [ ]:
from pathlib import Path

# Tìm thư mục dự án từ vị trí đang mở notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "Datasets/data_raw/spam_ham_dataset.csv").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Hãy mở notebook bên trong thư mục Present_and_Lab1_ML_G5.")
    ROOT = ROOT.parent

print("Thư mục dự án:", ROOT)


## 2. Đọc và làm sạch dữ liệu
Hàm `clean_text` được viết trực tiếp trong ô code bên dưới: chuyển chữ thường, bỏ HTML/dấu câu, gộp khoảng trắng. Phân loại văn bản email tiếng Anh, không phân tích file đính kèm.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Hàm làm sạch 
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

# 1. Đọc dữ liệu: chỉ lấy văn bản và nhãn.
data = pd.read_csv(ROOT / "Datasets/data_raw/spam_ham_dataset.csv")
data = data[["text", "label"]].dropna().copy()
print("Trước xử lý:", len(data))

# 2. Làm sạch và loại nội dung trùng trước khi chia tập.
data["clean_text"] = data["text"].apply(clean_text)
data = data[data["clean_text"] != ""]
if data.groupby("clean_text")["label"].nunique().max() > 1:
    raise ValueError("Có nội dung trùng nhưng khác nhãn.")
data = data.drop_duplicates(subset="clean_text")
print("Sau xử lý:", len(data))
print(data["label"].value_counts())
data[["label", "clean_text"]].head()


## 3. Chia train/test
80% để học, 20% để kiểm tra. `random_state=42` giữ lần chia cố định. `stratify` giữ tỷ lệ nhãn gần nhau.

In [ ]:
# 3. Chia văn bản trước khi học TF-IDF.
text_train, text_test, y_train, y_test = train_test_split(
    data["clean_text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)
print("Train:", len(text_train), "Test:", len(text_test))


## 4. Chuyển văn bản thành số
TF-IDF biểu diễn mỗi email bằng trọng số từ. `stop_words` bỏ từ thông dụng tiếng Anh; `max_features=1000` giới hạn số cột. Dùng bước này trước khi train Naive Bayes. Không fit trên test.

In [ ]:
# 4. Học từ vựng và trọng số từ train, chỉ biến đổi test.
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 5. Tự viết class Naive Bayes
`__init__` khai báo biến; `build_model` khởi tạo nơi lưu xác suất;
`fit` tính xác suất lớp và đặc trưng; `predict` tính điểm và chọn nhãn.
Không gọi MultinomialNB. NumPy chỉ hỗ trợ mảng, tổng và logarithm.

Với lớp c: `P(c) = số email lớp c / tổng email train`.
Với đặc trưng j: `P(j|c) = (S[c,j] + alpha) / (tổng_j S[c,j] + alpha * V)`.
S là tổng trọng số TF-IDF, V là số đặc trưng; alpha=1 là làm trơn.
Điểm: `log P(c) + tổng_j x[j] * log P(j|c)`; chọn lớp có điểm lớn nhất.
Dùng log để tránh tích các số rất nhỏ. TF-IDF không phải số đếm nguyên.
Chi tiết xem [implementation](../docs/implementation.md).


In [ ]:
import numpy as np

class NaiveBayesSpamClassifier:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.prior = None
        self.word_probability = None

    def build_model(self):
        if self.alpha <= 0:
            raise ValueError("Alpha phải lớn hơn 0.")
        self.classes = None
        self.prior = {}
        self.word_probability = {}

    def fit(self, X_train, y_train):
        if self.prior is None:
            raise ValueError("Hãy gọi build_model() trước fit().")
        y_train = np.array(y_train)
        self.classes = np.unique(y_train)
        number_of_emails, number_of_words = X_train.shape

        for label in self.classes:
            # Chọn các email thuộc lớp đang xét.
            X_class = X_train[y_train == label]
            self.prior[label] = X_class.shape[0] / number_of_emails

            # Tổng trọng số TF-IDF của mỗi từ trong lớp.
            word_totals = np.asarray(X_class.sum(axis=0)).ravel()
            self.word_probability[label] = (word_totals + self.alpha) / (
                word_totals.sum() + self.alpha * number_of_words
            )

    def predict(self, X_test):
        if self.classes is None:
            raise ValueError("Hãy gọi fit() trước predict().")
        predictions = []
        for row in range(X_test.shape[0]):
            email = X_test[row]
            scores = []
            for label in self.classes:
                # Tổng x[j] * log P(từ j | lớp), cộng log xác suất lớp.
                word_score = email.dot(np.log(self.word_probability[label])).item()
                score = np.log(self.prior[label]) + word_score
                scores.append(score)
            best_class = np.argmax(scores)
            predictions.append(self.classes[best_class])
        return np.array(predictions)


## 6. Bốn bước theo ghi chép của cô
Tạo đối tượng → build_model → fit → predict. Không gọi fit ngay sau khởi tạo.
Xem thêm docs/implementation.md.

In [ ]:
model = NaiveBayesSpamClassifier()  # 1. Khởi tạo
model.build_model()                 # 2. Tạo mô hình
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 7. Đánh giá — bài giảng trang 37–42
Spam là lớp dương. Ma trận có hàng là nhãn thật, cột là dự đoán, thứ tự ham/spam.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix [ham, spam]:")
print(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))

## 8. Dự đoán email mới
Sử dụng model và vectorizer vừa học ngay trong notebook, không lưu/nạp file mô hình. Hai email dưới là ví dụ minh họa, không thuộc tập đánh giá.

In [ ]:
emails = ["Subject: Team meeting tomorrow. Please review the report.",
          "Subject: Free prize. Claim your money now."]
clean_emails = [clean_text(email) for email in emails]
X_new = vectorizer.transform(clean_emails)
predictions = model.predict(X_new)
for email, label in zip(emails, predictions):
    print(label, "->", email)